# BERTScore for every summary (450 summaries)

Runtime → Change runtime type → **T4 GPU** (CPU also works, about 15 min). Then run all cells; it takes about 3 minutes on a GPU.

It reads only your existing Drive files (`summaries.csv`, `silver_references.csv`, `summaries_v2.csv`) and writes one new file, `outputs/step2/bertscore_all.csv`, then downloads it. Send that file back.

Settings are the ones recorded in run_config_step2.json: bert-score 0.3.13 with roberta-large. It saves both raw F1 (the setting used before) and baseline-rescaled F1, which spreads the scores out and is easier to read.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/hinglish_asr_sum'
!pip install -q bert-score==0.3.13

In [ ]:
import pandas as pd, torch, os
from bert_score import score
S2 = f'{ROOT}/outputs/step2'
s = pd.read_csv(f'{S2}/summaries.csv', keep_default_na=False)[['window_id','condition','out_lang','summary']]
if os.path.exists(f'{S2}/summaries_v2.csv'):
    v2 = pd.read_csv(f'{S2}/summaries_v2.csv', keep_default_na=False)[['window_id','condition','summary']]
    v2['out_lang'] = 'en'
    s = pd.concat([s, v2], ignore_index=True)
else:
    print('summaries_v2.csv not found - script-shift summaries skipped')
ref = pd.read_csv(f'{S2}/silver_references.csv', keep_default_na=False).set_index('window_id').ref_summary
cands, refs = s.summary.tolist(), [ref[w] for w in s.window_id]
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
P, R, F = score(cands, refs, model_type='roberta-large', lang='en', batch_size=16, device=dev)
_, _, Fr = score(cands, refs, model_type='roberta-large', lang='en', batch_size=16, device=dev, rescale_with_baseline=True)
s['bertscore_p'], s['bertscore_r'], s['bertscore_f1'] = P.tolist(), R.tolist(), F.tolist()
s['bertscore_f1_rescaled'] = Fr.tolist()
out = s.drop(columns=['summary'])
out.to_csv(f'{S2}/bertscore_all.csv', index=False)
print(out.groupby(['out_lang','condition'])[['bertscore_f1','bertscore_f1_rescaled']].mean().round(4))

In [ ]:
from google.colab import files
files.download(f'{S2}/bertscore_all.csv')